In [ ]:
from IPython.display import Markdown, display


In [ ]:
from src.retrieval.vector_store import VectorStore

vs = VectorStore()
print("ChromaDB connected")

In [ ]:
from pathlib import Path

from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import load_and_chunk_pdf
from src.ingestion.embedder import embed_texts

# Resolve path from project root (notebook often runs with cwd = notebooks/)
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent
pdf_path = ROOT / "uploaded_pdfs" / "Attention_is _all_you_need.pdf"

# parse and chunk
chunks = load_and_chunk_pdf(str(pdf_path))
print(f"Total chunks: {len(chunks)}")

# embed
vectors = embed_texts(chunks)
print(f"Total vectors: {len(vectors)}")

# save to ChromaDB
vs.add(chunks, vectors, "attention")
print("Saved to ChromaDB")

In [ ]:
vectors = embed_texts(chunks)
print(f"Total vectors: {len(vectors)}")


This **.add** will add the pdf in VS by passsing the hash function i wrote later on in this project so we still can have duplicates if we add manually take care of that issue if you notice.

In [ ]:
vs.add(chunks, vectors, "attention")
print("Saved to ChromaDB")

In [ ]:
from src.ingestion.embedder import embed_texts

query = "What is the attention mechanism?"
query_vector = embed_texts([query])[0]

results = vs.search(query_vector, top_k=5)

for i, result in enumerate(results):
    print(f"Result {i+1}:")
    print(result["text"][:200])
    print(f"Score: {result['score']}")
    print("---")

In [ ]:
from pathlib import Path
from src.retrieval.vector_store import VectorStore
from src.ingestion.pdf_parser import parse_pdf
from src.ingestion.chunker import load_and_chunk_pdf
from src.ingestion.embedder import embed_texts

# resolve project root
ROOT = Path.cwd()
if not (ROOT / "uploaded_pdfs").exists():
    ROOT = ROOT.parent

pdf_path = ROOT / "uploaded_pdfs" / "Attention_is _all_you_need.pdf"

# parse and chunk
print("Parsing and chunking PDF...")
chunks = load_and_chunk_pdf(str(pdf_path))
print(f"Total chunks: {len(chunks)}")

# embed
print("Embedding chunks...")
vectors = embed_texts(chunks)
print(f"Total vectors: {len(vectors)}")

# save to ChromaDB
print("Saving to ChromaDB...")
vs = VectorStore()
vs.add(chunks, vectors, "attention")
print("Saved to ChromaDB ✅")

# test search
print("\nTesting search...")
query = "What is the attention mechanism?"
query_vector = embed_texts([query])[0]
results = vs.search(query_vector, top_k=5)

for i, result in enumerate(results):
    print(f"\nResult {i+1} (score: {result['score']:.4f}):")
    print(result["text"][:200])
    print("---")

Testing the pipeline

1st paper

In [ ]:
from src.ingestion.ingest_pipeline import run_ingestion

run_ingestion("uploaded_pdfs/Attention_is _all_you_need.pdf")

2nd paper

In [ ]:
from src.ingestion.ingest_pipeline import run_ingestion

run_ingestion("uploaded_pdfs/AGENTIC RETRIEVAL-AUGMENTED GENERATION- A SURVEY ON AGENTIC RAG.pdf")

3rd paper

In [ ]:
from src.ingestion.ingest_pipeline import run_ingestion

run_ingestion("uploaded_pdfs/Mixture-of-Agents Enhances Large Language Model Capabilities.pdf")

4th paper

In [ ]:
from src.ingestion.ingest_pipeline import run_ingestion

run_ingestion("uploaded_pdfs/TinyLlama- An Open-Source Small Language Model.pdf")

We need to check how many papers were in the vector database

In [ ]:
from src.retrieval.vector_store import VectorStore

vs = VectorStore()
print("Papers:", vs.list_papers())
print("Number of papers:", vs.count_papers())

We got 5 papers because the papers are distinguished with the paper name (Need to chnage this later in the project to remove duplicates automatically.)

In [ ]:
from src.retrieval.vector_store import VectorStore
vs = VectorStore()
vs.list_papers()  # see the actual names

Removing that duplicate.

In [ ]:
vs.delete("attention")

Now we have only 4 papers.

In [ ]:
from src.retrieval.vector_store import VectorStore
vs = VectorStore()
vs.list_papers()  # see the actual names

Testing wether the duplicate detcion is working or not.

In [ ]:
from src.ingestion.ingest_pipeline import run_ingestion

result = run_ingestion("uploaded_pdfs/Attention_is _all_you_need.pdf")
print(result)

The Duplicates removal has worked for now 

Adding all the papers in the uploaded_pdfs. 

 **Dont run this multiple time it might cost you more because each time you add a duplicated it will delete the old one and replaces it with this one**

In [ ]:
# --- Load-all-papers cell: commented so we don't re-run ingestion ---
# from pathlib import Path
#
# from src.ingestion.ingest_pipeline import run_ingestion
#
# # Resolve project root (notebook often runs with cwd = notebooks/)
# ROOT = Path.cwd()
# if not (ROOT / "uploaded_pdfs").exists():
#     ROOT = ROOT.parent
# pdf_dir = ROOT / "uploaded_pdfs"
#
# # Ingest every PDF in uploaded_pdfs into the vector store
# pdf_paths = sorted(pdf_dir.glob("*.pdf"))
# print(f"Found {len(pdf_paths)} PDFs in uploaded_pdfs/")
#
# for pdf_path in pdf_paths:
#     stats = run_ingestion(pdf_path)
#     print(f"  {pdf_path.name}: {stats['num_chunks']} chunks")
#
# print("Done. Papers in VS:", vs.list_papers())

In [ ]:
from src.retrieval.retriever import retrieve

results = retrieve("What is the attention mechanism?")
print(f"Total results: {len(results)}")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is this project?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("What is paris located in the world?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("Does agentic retrieval works?")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what are the uses of RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")

In [ ]:
results = retrieve("what is RAG")

for i, result in enumerate(results[:5]):
    print(f"Result {i+1} (score: {result['score']:.4f}):")
    display(Markdown(result["text"]))
    print("---")